# Notebook 18c — NSV Stage 1 with 10-Day Prediction Lag (BSISO MJJAS, lp25)
**Project:** ENSO-BSISO SSL — Neural State Variables extension  
**Author:** Jiayi (jh9141@nyu.edu)

Third Stage 1 attempt. **nb18** (Lee, lag=1) and **nb18b** (lp25, lag=1) both failed to produce a manifoldized latent, for opposite reasons:

| | nb18 (Lee, lag=1) | nb18b (lp25, lag=1) | nb18c (lp25, **lag=10**) |
|---|:-:|:-:|:-:|
| Persistence MSE | 1.231 | **0.004** | (this notebook) |
| Model val MSE | 0.892 | 0.140 | (this notebook) |
| Model vs persistence | 27.5% better | **3,400% worse** | TBD |
| PC1 of latent | 6.5% | 9.7% | TBD (target > 25%) |
| Failure mode | encoder absorbs synoptic noise | persistence trivializes prediction; encoder reduces to lossy autoencoder | — |

**The lag=10 fix.** With lp25 the field changes by `~0.004` per day. The 64-D bottleneck loses `~0.14` per reconstruction. So at lag=1 the bottleneck error swamps the dynamics signal — the encoder gets no useful gradient pressure to compress state, and the latent stays diffuse across 64 dims.

At lag=10 the field has changed enough that *only knowing where in the BSISO cycle today sits* helps predict 10 days ahead. Fast small-scale variations are uncorrelated at 10-day lag — the encoder is structurally forced to extract the **slow BSISO state** and discard the rest. This is exactly the regime NSV needs.

## Why lag = 10 specifically

BSISO has a 30–60 d period. Useful lag windows:

| Lag (days) | Fraction of BSISO cycle | Expected behavior |
|:-:|:-:|---|
| 1 | 2–3% | Trivially predicted by persistence (nb18b case) |
| 5 | 8–17% | Persistence still close; weak state signal |
| **10** | **17–33%** | **Persistence becomes nontrivial; phase information becomes decisive** |
| 15 | 25–50% | Phase information dominant but fewer pairs |
| 20 | 33–67% | Approaching decorrelation; risk of unpredictability |

10 days is the sweet spot — far enough to break persistence, close enough that the slow BSISO state is still strongly predictive.

## Architecture and hyperparameters

**Identical to nb18b.** Same `EncoderBSISO` (5 conv blocks → 64-D bottleneck), same `DecoderBSISO` (bilinear-upsample + conv pyramid), same Adam-lr1e-3-cosine schedule, same MSE loss, same batch 64, same 100 epochs. The only change is what the encoder is asked to predict.

## Inputs

Reads from `/processed/` directly (not nb17b's lag-1 pair outputs):
- `X_MJJAS_lee_lp25.npy` shape `(4386, 3, 31, 51)` (43 years × 102 days/year after lp25 edge drop)
- `labels_aligned_mjjas_lee_lp25.csv` — for dates + BSISO phase + ENSO category

## Outputs (`BSISO_SSL_Project/nsv/`)

- `checkpoints_lag10/encoder_stage1.pth`, `decoder_stage1.pth`, `*_best.pth`, `training_history_stage1.json`
- `latents_lag10/z_train.npy`, `z_val.npy`
- `results/stage1_lag10/training_curves.png`, `reconstructions.png`, `latent_diagnostics.png`, `stage1_summary.json`

## Expected pair count

Each MJJAS lp25 year has ~103 days; at lag=10, anchors run from day 1 to day 93 → **93 pairs/year × 43 years ≈ 3,999 pairs**. Slightly less than the 4,386 of lag=1 but still healthy: `log₂(3999) ≈ 12`, so Levina-Bickel remains reliable up to d ≈ 10.

## Verification gate

1. **Encoder beats persistence by a meaningful margin** (≥ 20% MSE reduction). On lag=10, persistence should be ~0.05–0.20 (10× to 50× higher than lag=1); model val MSE should beat that.
2. **PC1 fraction > 25%** in the latent — the manifold-structure target. If still flat-isotropic, lag=10 wasn't enough either.
3. **Reconstructions** look spatially coherent vs the lag-10 targets, not blurred constants.

## Runtime

~3 min on Colab T4 (same model size, slightly fewer pairs than nb18b).

---

## Cell 1 — Mount Drive, Load X_MJJAS_lee_lp25 Directly, Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
NSV_DIR       = f'{PROJECT_DIR}/nsv'
CKPT_DIR      = f'{NSV_DIR}/checkpoints_lag10'
LATENT_DIR    = f'{NSV_DIR}/latents_lag10'
RESULTS_DIR   = f'{NSV_DIR}/results/stage1_lag10'
for d in [CKPT_DIR, LATENT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Hyperparameters — identical to nb18b except LAG_DAYS
LAG_DAYS     = 10           # ← the key change vs nb18b (which had implicit lag=1)
LATENT_DIM   = 64
BATCH_SIZE   = 64
EPOCHS       = 100
LR           = 1e-3
WEIGHT_DECAY = 1e-4
SEED         = 42
VAL_YEARS_STRIDE = 5

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:    {device}')
print(f'Lag:       {LAG_DAYS} days')
print(f'Latent dim: {LATENT_DIM}  (overparameterized; true BSISO ID expected 2-5)')

# Load the FULL day-indexed lp25 data (not nb17b's lag-1 pair tensors)
X      = np.load(f'{PROCESSED_DIR}/X_MJJAS_lee_lp25.npy')
labels = pd.read_csv(f'{PROCESSED_DIR}/labels_aligned_mjjas_lee_lp25.csv', parse_dates=['date'])
labels['date'] = labels['date'].dt.normalize()
assert X.shape[0] == len(labels), f'X / labels mismatch: {X.shape[0]} vs {len(labels)}'
assert X.shape[1:] == (3, 31, 51), f'Unexpected X shape {X.shape[1:]}'
print(f'Loaded {X.shape[0]} lp25 days  ({labels["date"].min().date()} to {labels["date"].max().date()}).')

## Cell 2 — Build Lag-10 Pairs + Year-Based Split + Persistence Baseline

Pair rule: `(X[i], X[i + LAG])` is valid iff `dates[i + LAG] − dates[i] == LAG days` (no gaps) AND `year(dates[i]) == year(dates[i + LAG])` (no winter jump). For lp25 data within each year's continuous May 26 – Sep 5 block, the same-year + exact-delta check is sufficient.

We also compute the lag-10 persistence baseline here — this is the number the model must beat.

In [ ]:
order = np.argsort(labels['date'].values)
if not np.array_equal(order, np.arange(len(labels))):
    X = X[order]; labels = labels.iloc[order].reset_index(drop=True)

dates_all = pd.DatetimeIndex(labels['date'].values)
years_all = dates_all.year.values
N_total = len(dates_all)

# Build lag-LAG pair indices
delta_days  = (dates_all[LAG_DAYS:] - dates_all[:-LAG_DAYS]).days
same_year   = years_all[LAG_DAYS:] == years_all[:-LAG_DAYS]
valid_start = (delta_days == LAG_DAYS) & same_year                          # length N_total - LAG_DAYS
valid_start = np.concatenate([valid_start, np.zeros(LAG_DAYS, dtype=bool)])  # pad to length N_total

pair_idx_t  = np.where(valid_start)[0]
pair_idx_t1 = pair_idx_t + LAG_DAYS
N_pairs = len(pair_idx_t)

# Hard assertion: deltas are exactly LAG and stay within year
actual_deltas = (dates_all[pair_idx_t1] - dates_all[pair_idx_t]).days
assert actual_deltas.min() == LAG_DAYS and actual_deltas.max() == LAG_DAYS, f'Deltas not all = {LAG_DAYS}'
assert (years_all[pair_idx_t1] == years_all[pair_idx_t]).all(), 'Year-boundary pair leaked through'

print(f'Pair construction:  {N_pairs} valid lag-{LAG_DAYS} pairs  '
      f'(expected ~{43 * (103 - LAG_DAYS)} = 43 yrs × {103 - LAG_DAYS} pairs/yr).')

# Build pair tensors (keep in RAM; ~200 MB total)
X_t  = X[pair_idx_t].astype(np.float32)
X_t1 = X[pair_idx_t1].astype(np.float32)
dates_t = dates_all[pair_idx_t]

# Year-based train/val split (every 5th year)
all_years   = sorted(np.unique(years_all).tolist())
val_years   = all_years[::VAL_YEARS_STRIDE]
train_years = sorted(set(all_years) - set(val_years))
pair_years  = dates_t.year.values
train_mask  = np.isin(pair_years, train_years)
n_train = int(train_mask.sum()); n_val = int((~train_mask).sum())
print(f'Train: {n_train} pairs from {len(train_years)} years  /  Val: {n_val} pairs from {len(val_years)} years {val_years}')

# Persistence baseline at lag-LAG (predict X_{t+LAG} = X_t)
X_t_val  = torch.from_numpy(X_t[~train_mask]).float()
X_t1_val = torch.from_numpy(X_t1[~train_mask]).float()
persistence_mse = ((X_t_val - X_t1_val) ** 2).mean().item()
print(f'\nPersistence MSE at lag={LAG_DAYS} (predict X_t+{LAG_DAYS} = X_t): {persistence_mse:.4f}')
print(f'  nb18b (Lee, lag=1):  1.231   — model achieved 0.892 → 27.5% better')
print(f'  nb18b (lp25, lag=1): 0.004   — model achieved 0.140 → 3,400% WORSE (persistence trivializes prediction)')
print(f'  this run (lp25, lag={LAG_DAYS}): {persistence_mse:.4f}   — model must beat this; informativeness is the comparison')

# Reusable labels for downstream notebooks
bsiso_phase_t      = labels['bsiso_phase'].values[pair_idx_t].astype(np.int8)
bsiso_amplitude_t  = labels['bsiso_amplitude'].values[pair_idx_t].astype(np.float32)
enso_cat_t         = labels['enso_category'].values[pair_idx_t].astype('<U10')

# Save pair metadata (latents and labels are what nb19/20 need; pair tensors stay in RAM only)
np.save(f'{LATENT_DIR}/dates_t.npy', dates_t.values.astype('datetime64[D]'))
np.save(f'{LATENT_DIR}/bsiso_phase_t.npy', bsiso_phase_t)
np.save(f'{LATENT_DIR}/bsiso_amplitude_t.npy', bsiso_amplitude_t)
np.save(f'{LATENT_DIR}/enso_cat_t.npy', enso_cat_t)
np.save(f'{LATENT_DIR}/train_mask.npy', train_mask)
print(f'\nSaved label arrays (aligned to lag-{LAG_DAYS} pair anchors) to {LATENT_DIR}/')

## Cell 3 — Encoder + Decoder (Identical to nb18 / nb18b)

Same architecture as both prior Stage 1 attempts. The only thing that's changing is what `X_t1` looks like (10 days ahead, not 1).

In [ ]:
class EncoderBSISO(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  32,  4, stride=2, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32,  3, stride=1, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64,  4, stride=2, padding=1, bias=False); self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64,  3, stride=1, padding=1, bias=False); self.bn4 = nn.BatchNorm2d(64)
        self.conv5 = nn.Conv2d(64, 128, 4, stride=2, padding=1, bias=False); self.bn5 = nn.BatchNorm2d(128)
        self.gap = nn.AdaptiveAvgPool2d(1); self.fc = nn.Linear(128, latent_dim)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):       nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d): nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):     nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = self.gap(x).flatten(1)
        return self.fc(x)

class DecoderBSISO(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128)
        self.conv1 = nn.Conv2d(128, 64, 3, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64,  64, 3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64,  32, 3, padding=1, bias=False); self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32,  3,  3, padding=1)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d): nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):     nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)
    def forward(self, z):
        x = self.fc(z).view(-1, 128, 1, 1)
        x = F.interpolate(x, size=(3, 6),   mode='bilinear', align_corners=False); x = F.relu(self.bn1(self.conv1(x)))
        x = F.interpolate(x, size=(7, 12),  mode='bilinear', align_corners=False); x = F.relu(self.bn2(self.conv2(x)))
        x = F.interpolate(x, size=(15, 25), mode='bilinear', align_corners=False); x = F.relu(self.bn3(self.conv3(x)))
        x = F.interpolate(x, size=(31, 51), mode='bilinear', align_corners=False)
        return self.conv4(x)

enc = EncoderBSISO(LATENT_DIM).to(device)
dec = DecoderBSISO(LATENT_DIM).to(device)
n_params = sum(p.numel() for p in enc.parameters()) + sum(p.numel() for p in dec.parameters())
with torch.no_grad():
    dummy = torch.randn(4, 3, 31, 51).to(device)
    z = enc(dummy); xhat = dec(z)
    assert z.shape == (4, LATENT_DIM) and xhat.shape == dummy.shape
    print(f'Total params: {n_params:,}  (same as nb18/nb18b).  Forward sanity OK.')

## Cell 4 — Dataset + DataLoaders

In [ ]:
class PairDataset(Dataset):
    def __init__(self, X_t, X_t1, indices):
        self.X_t  = torch.from_numpy(X_t[indices]).float()
        self.X_t1 = torch.from_numpy(X_t1[indices]).float()
    def __len__(self):  return self.X_t.shape[0]
    def __getitem__(self, k):  return self.X_t[k], self.X_t1[k]

train_idx_arr = np.where(train_mask)[0]
val_idx_arr   = np.where(~train_mask)[0]
train_ds = PairDataset(X_t, X_t1, train_idx_arr)
val_ds   = PairDataset(X_t, X_t1, val_idx_arr)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds):5d} pairs  → {len(train_loader)} batches/epoch')
print(f'Val:   {len(val_ds):5d} pairs  → {len(val_loader)} batches/epoch')

## Cell 5 — Training Loop (Identical to nb18 / nb18b)

In [ ]:
from tqdm.notebook import tqdm

params = list(enc.parameters()) + list(dec.parameters())
optimizer = optim.Adam(params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

history = {'train_mse': [], 'val_mse': [], 'epoch_time': []}
best_val = float('inf')

for epoch in range(EPOCHS):
    t0 = time.time()
    enc.train(); dec.train()
    train_loss = 0.0; n_seen = 0
    pbar = tqdm(train_loader, desc=f'ep {epoch+1}/{EPOCHS}', leave=False)
    for x_t, x_t1 in pbar:
        x_t  = x_t.to(device, non_blocking=True); x_t1 = x_t1.to(device, non_blocking=True)
        loss = F.mse_loss(dec(enc(x_t)), x_t1)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        train_loss += loss.item() * x_t.size(0); n_seen += x_t.size(0)
        pbar.set_postfix({'mse': f'{loss.item():.4f}'})
    train_mse = train_loss / n_seen

    enc.eval(); dec.eval()
    val_loss = 0.0; n_seen = 0
    with torch.no_grad():
        for x_t, x_t1 in val_loader:
            x_t  = x_t.to(device, non_blocking=True); x_t1 = x_t1.to(device, non_blocking=True)
            val_loss += F.mse_loss(dec(enc(x_t)), x_t1, reduction='sum').item() / x_t1.numel() * x_t.size(0)
            n_seen += x_t.size(0)
    val_mse = val_loss / n_seen
    scheduler.step()

    et = time.time() - t0
    history['train_mse'].append(train_mse); history['val_mse'].append(val_mse); history['epoch_time'].append(et)
    print(f'ep {epoch+1:3d}/{EPOCHS}  train={train_mse:.4f}  val={val_mse:.4f}  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  time={et:.1f}s')

    if val_mse < best_val:
        best_val = val_mse
        torch.save(enc.state_dict(), f'{CKPT_DIR}/encoder_stage1_best.pth')
        torch.save(dec.state_dict(), f'{CKPT_DIR}/decoder_stage1_best.pth')

torch.save(enc.state_dict(), f'{CKPT_DIR}/encoder_stage1.pth')
torch.save(dec.state_dict(), f'{CKPT_DIR}/decoder_stage1.pth')
with open(f'{CKPT_DIR}/training_history_stage1.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nDone in {sum(history["epoch_time"])/60:.1f} min.')
print(f'Best val MSE:    {best_val:.4f}  (at epoch {history["val_mse"].index(best_val)+1}).')
print(f'Final train MSE: {history["train_mse"][-1]:.4f}    val MSE: {history["val_mse"][-1]:.4f}')
print(f'Persistence (lag={LAG_DAYS}): {persistence_mse:.4f}')
ratio = (persistence_mse - best_val) / persistence_mse * 100
print(f'Improvement over persistence: {ratio:+.1f}%  (target > +20% for a useful encoder)')

## Cell 6 — Diagnostics + Latent Extraction

Same three diagnostics as nb18b Cell 5 (training curves, reconstructions, PCA scree + per-dim std), followed by latent extraction. The **PC1 fraction** on the scree subplot is the single decisive number — if it jumps to >25% the manifold appeared; if it's still ~10% lag=10 wasn't enough.

In [ ]:
enc.load_state_dict(torch.load(f'{CKPT_DIR}/encoder_stage1_best.pth', map_location=device))
dec.load_state_dict(torch.load(f'{CKPT_DIR}/decoder_stage1_best.pth', map_location=device))
enc.eval(); dec.eval()

# 1) Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(history['train_mse'], label='Train MSE', lw=2)
axes[0].plot(history['val_mse'],   label='Val MSE',   lw=2)
axes[0].axhline(persistence_mse, color='gray', ls='--', lw=1, label=f'Persistence lag={LAG_DAYS} ({persistence_mse:.3f})')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].set_title(f'Training Curves (lp25, lag={LAG_DAYS})', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['epoch_time'], color='green', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Time (s)')
axes[1].set_title('Per-Epoch Time', fontweight='bold')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=140, bbox_inches='tight')
plt.show()

# 2) Reconstructions on 4 random val pairs
rng = np.random.default_rng(SEED)
picks = rng.choice(len(val_ds), size=4, replace=False)
x_t_pick  = val_ds.X_t[picks].to(device); x_t1_pick = val_ds.X_t1[picks].to(device)
with torch.no_grad():
    x_t1_hat = dec(enc(x_t_pick)).cpu().numpy()
x_t_np  = x_t_pick.cpu().numpy(); x_t1_np = x_t1_pick.cpu().numpy()

olr_ch = 2
fig, axes = plt.subplots(4, 3, figsize=(13, 14), sharex=True, sharey=True)
fig.suptitle(f"lp25 lag={LAG_DAYS} val reconstructions (OLR): X_t (left) | X_t+{LAG_DAYS} target | X̂_t+{LAG_DAYS} predicted",
             fontsize=12, fontweight='bold')
vmax = max(np.abs(x_t_np[:, olr_ch]).max(), np.abs(x_t1_np[:, olr_ch]).max(), np.abs(x_t1_hat[:, olr_ch]).max())
for r in range(4):
    for c, (arr, label) in enumerate([(x_t_np, 'X_t'), (x_t1_np, f'X_t+{LAG_DAYS} target'), (x_t1_hat, f'X̂_t+{LAG_DAYS} predicted')]):
        ax = axes[r, c]
        im = ax.imshow(arr[r, olr_ch], cmap='RdBu_r', aspect='auto',
                       extent=[60, 160, 0, 60], vmin=-vmax, vmax=vmax, origin='lower')
        if r == 0: ax.set_title(label, fontsize=11, fontweight='bold')
        if c == 0: ax.set_ylabel(f'pair {picks[r]}', fontsize=10)
        if r == 3: ax.set_xlabel('Longitude (°)')
fig.subplots_adjust(right=0.90)
cb = fig.add_axes([0.92, 0.15, 0.015, 0.7])
plt.colorbar(im, cax=cb, label="OLR' (σ)")
plt.savefig(f'{RESULTS_DIR}/reconstructions.png', dpi=130, bbox_inches='tight')
plt.show()

# 3) Latent diagnostics — PCA scree is the headline number
with torch.no_grad():
    z_train_diag = np.concatenate([enc(train_ds.X_t[k:k+256].to(device)).cpu().numpy()
                                    for k in range(0, len(train_ds), 256)], axis=0)
z_std = z_train_diag.std(axis=0)
n_active = int((z_std > 0.01).sum())
from sklearn.decomposition import PCA
pca_diag = PCA(n_components=min(15, LATENT_DIM)).fit(z_train_diag - z_train_diag.mean(0))
var_ratio_diag = pca_diag.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
ax = axes[0]
ax.bar(np.arange(LATENT_DIM), z_std, color='steelblue', alpha=0.85)
ax.axhline(z_std.mean(), color='red', ls='--', lw=1, label=f'Mean std = {z_std.mean():.3f}')
ax.axhline(0.01, color='gray', ls=':', lw=1, label='Collapse threshold')
ax.set_xlabel('Latent dim'); ax.set_ylabel('std across train pairs')
ax.set_title(f'Per-Dim std  (active: {n_active}/{LATENT_DIM};  nb18 & nb18b: 64/64 uniform ~0.031)',
             fontweight='bold', fontsize=11)
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.bar(range(1, len(var_ratio_diag)+1), var_ratio_diag*100, color='steelblue', alpha=0.85)
ax.set_xlabel('PC index'); ax.set_ylabel('% variance')
ax.set_title(f'PCA scree (first {len(var_ratio_diag)} PCs)\n'
             f'PC1 = {var_ratio_diag[0]*100:.1f}%   '
             f'(nb18 = 6.5%, nb18b = 9.7%; target > 25% for a real manifold)',
             fontweight='bold', fontsize=11)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/latent_diagnostics.png', dpi=140, bbox_inches='tight')
plt.show()

print(f'\nLatent diagnostics (lag={LAG_DAYS}):')
print(f'  active dims:  {n_active}/{LATENT_DIM}')
print(f'  per-dim std:  min={z_std.min():.4f}, max={z_std.max():.4f}, mean={z_std.mean():.4f}')
print(f'  PC1 / PC2:    {var_ratio_diag[0]*100:.2f}%  /  {var_ratio_diag[1]*100:.2f}%')
print(f'  cum at 5 PCs: {np.cumsum(var_ratio_diag)[4]*100:.2f}%')
if var_ratio_diag[0] > 0.25:
    print('✓ PC1 > 25% — manifold appears! Proceed to nb19 on lag-10 latents.')
elif var_ratio_diag[0] > 0.15:
    print('△ PC1 between 15–25% — partial manifoldization. Proceed to nb19 but expect modest confidence.')
else:
    print('✗ PC1 < 15% — manifold still diffuse. Consider longer lag or smaller bottleneck.')

# 4) Extract & save latents for nb19
def extract_z(dataset, encoder, batch=256):
    encoder.eval()
    with torch.no_grad():
        return np.concatenate([encoder(dataset.X_t[k:k+batch].to(device)).cpu().numpy()
                                for k in range(0, len(dataset), batch)], axis=0).astype(np.float32)

z_train = extract_z(train_ds, enc)
z_val   = extract_z(val_ds,   enc)
np.save(f'{LATENT_DIR}/z_train.npy', z_train)
np.save(f'{LATENT_DIR}/z_val.npy',   z_val)
print(f'\nSaved latents: z_train {z_train.shape}, z_val {z_val.shape}  →  {LATENT_DIR}/')

stage1_summary = {
    'variant':             f'lp25 + lag={LAG_DAYS}',
    'lag_days':            LAG_DAYS,
    'latent_dim':          LATENT_DIM,
    'epochs':              EPOCHS,
    'n_pairs_total':       N_pairs,
    'n_pairs_train':       n_train,
    'n_pairs_val':         n_val,
    'best_val_mse':        float(best_val),
    'final_train_mse':     float(history['train_mse'][-1]),
    'final_val_mse':       float(history['val_mse'][-1]),
    'persistence_mse':     float(persistence_mse),
    'improvement_pct':     float((persistence_mse - best_val) / persistence_mse * 100),
    'n_active_dims':       n_active,
    'pc1_var_ratio':       float(var_ratio_diag[0]),
    'pc2_var_ratio':       float(var_ratio_diag[1]),
    'cum_var_5pcs':        float(np.cumsum(var_ratio_diag)[4]),
}
with open(f'{RESULTS_DIR}/stage1_summary.json', 'w') as f:
    json.dump(stage1_summary, f, indent=2)
print(f'Saved summary: {RESULTS_DIR}/stage1_summary.json')
print('\n→ To run nb19 on these latents:')
print(f'   LATENT_DIR  = f\'{{NSV_DIR}}/latents_lag10\'')
print(f'   RESULTS_DIR = f\'{{NSV_DIR}}/results/stage2_lag10\'')

---
## Done!

**Send back** for review:
1. Cell 5 final lines — `Best val MSE` vs `Persistence (lag=10)` and the improvement %.
2. `results/stage1_lag10/training_curves.png` — train+val should descend toward and beat the dashed persistence line.
3. `results/stage1_lag10/latent_diagnostics.png` — **the headline figure**. PC1 % on the right subplot is the single decisive number.
4. `results/stage1_lag10/reconstructions.png` — predictions should look like spatially coherent BSISO patterns, not blurred constants.
5. `results/stage1_lag10/stage1_summary.json` — top-line numbers.

**Outcomes to expect:**

| PC1 fraction | What it means | Next |
|---|---|---|
| **> 25%** | ✓ Manifold appeared. lag=10 forced state extraction. | Run nb19 on `latents_lag10/` — expect d̂ in 2–5 range with HIGH confidence. |
| 15–25% | △ Partial manifoldization. | Run nb19; d̂ may have MEDIUM confidence. Could try lag=15 if d̂ is still saturated. |
| < 15% | ✗ Lag wasn't enough. | Try lag=15 or 20, or reduce LATENT_DIM to force compression. |

---
*DDCS Project | jh9141@nyu.edu*